In [11]:

from dotenv import load_dotenv
from sentence_transformers import SentenceTransformer
from ollama import Client
import os
import numpy as np
from pydantic import BaseModel
load_dotenv()

OLLAMA_API_KEY = os.environ.get("OLLAMA_CLOUD_API_KEY")
HF_TOKEN = os.environ.get("HF_TOKEN")
client = Client(
    host="https://ollama.com",
    headers={'Authorization': 'Bearer ' + OLLAMA_API_KEY}
)

class Message(BaseModel):
    name: list[str]

messages = [
    "Apple is a tech company",
    "Meta is a tech company",
    "Reliance is not a tech company",
    "Bank Of England is not a tech company",
    "Google is a tech company"
]

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", token=HF_TOKEN)


embeddings = model.encode(messages, normalize_embeddings=True)

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

query_embedding = model.encode("Which companies are tech companies?", normalize_embeddings=True)

scores = []
for embedding in embeddings:
    score = cosine_similarity(query_embedding, embedding)
    scores.append(score)

scores.sort(reverse=True)
print(scores)

if scores[0] > 0.5:
    print("The query is related to tech companies.")
else:
    print("The query is not related to tech companies.")
    
top_k = 5
top_k_indices = np.argsort(scores)[-top_k:][::-1]
print("Top K relevant messages:", top_k_indices)
for idx in top_k_indices:
    print(messages[idx], "with score:", scores[idx])

schema=Message.model_json_schema(),
response = client.chat(    
    format="json",
    model="gpt-oss:120b",
    messages=[
        {"role": "user", "content": f"""Use the following context to answer the user's question. 
                                        Context: {[messages[idx] for idx in top_k_indices]}
                                        Question: Which companies are tech companies?
                                        Answer in JSON format according to the following schema: {schema}"""}],
    
)
print("Response from Ollama:", response.message.content)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 975.15it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[np.float32(0.664698), np.float32(0.6058529), np.float32(0.5832157), np.float32(0.53746444), np.float32(0.5284182)]
The query is related to tech companies.
Top K relevant messages: [0 1 2 3 4]
Apple is a tech company with score: 0.664698
Meta is a tech company with score: 0.6058529
Reliance is not a tech company with score: 0.5832157
Bank Of England is not a tech company with score: 0.53746444
Google is a tech company with score: 0.5284182
Response from Ollama: {
  "name": [
    "Apple",
    "Meta",
    "Google"
  ]
}
